# 🏆 Customer Lifetime Value (CLTV) Prediction System
## Phase 6: RFM Customer Segmentation
**Project**: Fortune 500 E-Commerce CLTV Prediction  
**Author**: Nisarga N  
**Date**: June 2026  
**Dataset**: UCI Online Retail II  
**Objective**: Build dynamic Recency, Frequency, Monetary (RFM) segments to drive targeted marketing actions.


In [ ]:
import sys
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to system path
sys.path.append(str(Path.cwd().parent))
from src.data_loader import DataLoader
from src.rfm_analyzer import RFMAnalyzer

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#c9d1d9',
    'text.color': '#c9d1d9', 'xtick.color': '#8b949e',
    'ytick.color': '#8b949e', 'grid.color': '#21262d',
    'font.size': 11, 'axes.titlesize': 14,
})

DATA_RAW = Path.cwd().parent / 'data' / 'raw'
DATA_PROCESSED = Path.cwd().parent / 'data' / 'processed'
VIZ_RFM = Path.cwd().parent / 'visualizations' / 'rfm'
VIZ_RFM.mkdir(parents=True, exist_ok=True)


## 1. Calculate Base RFM Values
We use the full dataset to compute recency, frequency, and monetary spend per customer.


In [ ]:
loader = DataLoader(DATA_RAW)
df = loader.load_processed(DATA_PROCESSED / "cleaned_retail.csv")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

ref_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

# Aggregate customer RFM metrics
rfm_base = df.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (ref_date - x.max()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("Revenue", "sum")
).reset_index()

rfm_base.head()


## 2. Execute RFM Segments Analysis
Instantiating our `RFMAnalyzer` object to score and classify customers.


In [ ]:
analyzer = RFMAnalyzer(rfm_base)
df_rfm, summary = analyzer.run_full_analysis()

# Save rfm segmentation results
df_rfm.to_csv(DATA_PROCESSED / "rfm_segments.csv", index=False)
print("Saved RFM segmentation results.")
summary


## 3. Visualize Segments
Visualizing customer counts across cohorts.


In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=summary, x="Customer_Count", y="Segment", palette="viridis")
plt.xlabel("Number of Customers")
plt.ylabel("Segment")
plt.title("Customer Segment Sizes")
plt.tight_layout()
plt.savefig(VIZ_RFM / "01_segment_sizes.png", dpi=150, facecolor='#0d1117')
plt.show()


### 💡 Business Insight
- A large segment of customers falls into `Champions` and `Loyal Customers`, suggesting a strong retention anchor.
- Reactivation campaigns should target `At Risk` customers to prevent high-value churn.
